In [1]:
# ===== Imports and settings =====
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RISK_ORDER = ["High", "Medium", "Low"]

def assign_risk(g3):
    if g3 <= 9:
        return "High"
    elif g3 <= 13:
        return "Medium"
    return "Low"

In [2]:
# ===== Pooled data with a subject indicator and student groups =====
mat = pd.read_csv("../data/raw/student-mat.csv", sep=";")
por = pd.read_csv("../data/raw/student-por.csv", sep=";")
mat["subject"] = 0
por["subject"] = 1

pooled = pd.concat([mat, por], ignore_index=True)
pooled["risk"] = pooled["G3"].apply(assign_risk)

keys = ["school", "sex", "age", "address", "famsize", "Pstatus",
        "Medu", "Fedu", "Mjob", "Fjob", "reason", "nursery", "internet"]
pooled["student_group"] = pooled.groupby(keys).ngroup()

# Engineered grade features
pooled["grade_change"] = pooled["G2"] - pooled["G1"]
pooled["grade_mean"] = (pooled["G1"] + pooled["G2"]) / 2
pooled["G1_x_subject"] = pooled["G1"] * pooled["subject"]
pooled["G2_x_subject"] = pooled["G2"] * pooled["subject"]

print("Rows:", len(pooled), " Student groups:", pooled["student_group"].nunique())

Rows: 1044  Student groups: 662


In [3]:
# ===== Final test set, split by student group and left untouched =====
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
dev_idx, final_idx = next(gss.split(pooled, pooled["risk"], pooled["student_group"]))

dev = pooled.iloc[dev_idx].reset_index(drop=True)
final_test = pooled.iloc[final_idx].reset_index(drop=True)
final_test.to_csv("../data/processed/final_test_do_not_open.csv", index=False)

print("Development:", len(dev), " Final test:", len(final_test))
print(dev["risk"].value_counts().reindex(RISK_ORDER).to_dict())
print(final_test["risk"].value_counts().reindex(RISK_ORDER).to_dict())

Development: 834  Final test: 210
{'High': 184, 'Medium': 412, 'Low': 238}
{'High': 46, 'Medium': 108, 'Low': 56}


In [4]:
# ===== Feature sets for the two stages =====
context = ["studytime", "failures", "absences", "famrel", "freetime",
           "goout", "Dalc", "Walc", "health", "traveltime"]

stages = {
    "Stage 1, G1 only": (["G1", "subject", "G1_x_subject"], "G1"),
    "Stage 1, G1 and context": (["G1", "subject", "G1_x_subject"] + context, "G1"),
    "Stage 2, G1 and G2": (["G1", "G2", "grade_change", "grade_mean",
                            "subject", "G1_x_subject", "G2_x_subject"], "G2"),
    "Stage 2, G1, G2 and context": (["G1", "G2", "grade_change", "grade_mean",
                                     "subject", "G1_x_subject", "G2_x_subject"] + context, "G2"),
}

models = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                        max_iter=200, random_state=RANDOM_STATE),
}

In [5]:
# ===== Cross validation and scoring helpers =====
def out_of_fold(X, y, groups, model, repeats=3):
    runs = []
    for seed in range(repeats):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + seed)
        pred = np.empty(len(y), dtype=object)
        for tr, te in cv.split(X, y, groups):
            pipe = Pipeline([("scale", StandardScaler()), ("clf", clone(model))])
            pipe.fit(X.iloc[tr], y.iloc[tr])
            pred[te] = pipe.predict(X.iloc[te])
        runs.append(pred)
    return runs


def score(y, pred, rule, mask):
    yy, pp = y[mask], pred[mask]
    ok_model = (pred == y) & mask
    ok_rule = (rule == y) & mask
    return {
        "recall_high": recall_score(yy, pp, labels=["High"], average="macro", zero_division=0),
        "precision_high": precision_score(yy, pp, labels=["High"], average="macro", zero_division=0),
        "f1_macro": f1_score(yy, pp, average="macro"),
        "accuracy": accuracy_score(yy, pp),
        "high_predicted_low": int(((y == "High") & (pred == "Low") & mask).sum()),
        "net_vs_rule": int((ok_model & ~ok_rule).sum() - (~ok_model & ok_rule).sum()),
    }

In [6]:
# ===== Run all stages, models and the rule baseline =====
y = dev["risk"]
groups = dev["student_group"]

subject_masks = {
    "All": np.ones(len(dev), dtype=bool),
    "Mathematics": (dev["subject"] == 0).values,
    "Portuguese": (dev["subject"] == 1).values,
}

y_arr = y.values
records = []

for stage, (cols, rule_col) in stages.items():
    X = dev[cols]
    rule_pred = dev[rule_col].apply(assign_risk).values

    for subj, mask in subject_masks.items():
        records.append({"stage": stage, "approach": "Rule, carry grade forward",
                        "subject": subj, **score(y_arr, rule_pred, rule_pred, mask)})

    for name, model in models.items():
        runs = out_of_fold(X, y, groups, model)
        for subj, mask in subject_masks.items():
            per_run = pd.DataFrame([score(y_arr, pred, rule_pred, mask) for pred in runs])
            records.append({"stage": stage, "approach": name,
                            "subject": subj, **per_run.mean().to_dict()})

results_pooled = pd.DataFrame(records).round(3)
results_pooled.to_csv("../outputs/tables/pooled_experiments.csv", index=False)
results_pooled[results_pooled["subject"] == "All"]

,stage,approach,subject,recall_high,precision_high,f1_macro,accuracy,high_predicted_low,net_vs_rule
0,"Stage 1, G1 only","Rule, carry grade forward",All,0.826,0.644,0.742,0.740,0.000,0.000
3,"Stage 1, G1 only",Logistic regression,All,0.777,0.673,0.730,0.727,0.000,-11.000
6,"Stage 1, G1 only",Random Forest,All,0.864,0.639,0.736,0.730,0.000,-8.333
9,"Stage 1, G1 only",Gradient boosting,All,0.719,0.707,0.743,0.748,0.000,7.000
12,"Stage 1, G1 and context","Rule, carry grade forward",All,0.826,0.644,0.742,0.740,0.000,0.000
15,"Stage 1, G1 and context",Logistic regression,All,0.801,0.658,0.734,0.729,0.000,-8.667
18,"Stage 1, G1 and context",Random Forest,All,0.777,0.682,0.740,0.738,1.000,-1.667
21,"Stage 1, G1 and context",Gradient boosting,All,0.694,0.724,0.737,0.741,0.333,1.333
24,"Stage 2, G1 and G2","Rule, carry grade forward",All,0.918,0.741,0.849,0.849,0.000,0.000
27,"Stage 2, G1 and G2",Logistic regression,All,0.895,0.779,0.841,0.839,0.000,-8.333


In [7]:
# ===== Look at the subjects separately =====
results_pooled[results_pooled["subject"] == "Portuguese"]

,stage,approach,subject,recall_high,precision_high,f1_macro,accuracy,high_predicted_low,net_vs_rule
2,"Stage 1, G1 only","Rule, carry grade forward",Portuguese,0.862,0.575,0.742,0.753,0.0,0.000
5,"Stage 1, G1 only",Logistic regression,Portuguese,0.729,0.648,0.744,0.757,0.0,2.333
8,"Stage 1, G1 only",Random Forest,Portuguese,0.737,0.618,0.736,0.749,0.0,-2.000
11,"Stage 1, G1 only",Gradient boosting,Portuguese,0.650,0.788,0.767,0.792,0.0,20.000
14,"Stage 1, G1 and context","Rule, carry grade forward",Portuguese,0.862,0.575,0.742,0.753,0.0,0.000
17,"Stage 1, G1 and context",Logistic regression,Portuguese,0.708,0.612,0.719,0.732,0.0,-10.667
20,"Stage 1, G1 and context",Random Forest,Portuguese,0.696,0.664,0.730,0.745,0.0,-4.333
23,"Stage 1, G1 and context",Gradient boosting,Portuguese,0.608,0.668,0.714,0.740,0.0,-6.667
26,"Stage 2, G1 and G2","Rule, carry grade forward",Portuguese,0.888,0.634,0.814,0.828,0.0,0.000
29,"Stage 2, G1 and G2",Logistic regression,Portuguese,0.833,0.739,0.822,0.830,0.0,1.000


In [8]:
# ===== Threshold free comparison for the High class =====
from sklearn.metrics import average_precision_score, precision_recall_curve

def oof_proba_high(X, y, groups, model, seed=0):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + seed)
    scores = np.zeros(len(y))
    for tr, te in cv.split(X, y, groups):
        pipe = Pipeline([("scale", StandardScaler()), ("clf", clone(model))])
        pipe.fit(X.iloc[tr], y.iloc[tr])
        proba = pipe.predict_proba(X.iloc[te])
        scores[te] = proba[:, list(pipe.classes_).index("High")]
    return scores


def precision_at_recall(is_high, score, target=0.90):
    prec, rec, _ = precision_recall_curve(is_high, score)
    return prec[rec >= target].max()


is_high = (dev["risk"] == "High").astype(int).values
rows = []

for stage in ["Stage 1, G1 only", "Stage 1, G1 and context", "Stage 2, G1 and G2"]:
    cols, rule_col = stages[stage]
    X = dev[cols]

    candidates = {"Rule score (lower grade is riskier)": [-dev[rule_col].values]}
    for name, model in models.items():
        candidates[name] = [oof_proba_high(X, y, groups, model, seed=s) for s in range(3)]

    for name, score_list in candidates.items():
        for subj, mask in subject_masks.items():
            rows.append({
                "stage": stage,
                "approach": name,
                "subject": subj,
                "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                          for s in score_list]),
                "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                                   for s in score_list]),
            })

results_curve = pd.DataFrame(rows).round(3)
results_curve.to_csv("../outputs/tables/pooled_threshold_free.csv", index=False)
results_curve

,stage,approach,subject,avg_precision,precision_at_90_recall
0,"Stage 1, G1 only",Rule score (lower grade is riskier),All,0.752,0.510
1,"Stage 1, G1 only",Rule score (lower grade is riskier),Mathematics,0.813,0.654
2,"Stage 1, G1 only",Rule score (lower grade is riskier),Portuguese,0.707,0.398
3,"Stage 1, G1 only",Logistic regression,All,0.777,0.622
4,"Stage 1, G1 only",Logistic regression,Mathematics,0.824,0.666
5,"Stage 1, G1 only",Logistic regression,Portuguese,0.716,0.462
6,"Stage 1, G1 only",Random Forest,All,0.770,0.619
7,"Stage 1, G1 only",Random Forest,Mathematics,0.799,0.655
8,"Stage 1, G1 only",Random Forest,Portuguese,0.689,0.456
9,"Stage 1, G1 only",Gradient boosting,All,0.768,0.619


In [9]:
# ===== Stronger simple baselines =====
def lr_scores(cols, seeds=range(3)):
    return [oof_proba_high(dev[cols], y, groups, models["Logistic regression"], seed=s)
            for s in seeds]

stage2_cols = stages["Stage 2, G1 and G2"][0]

candidates = {
    "Rule, G2 only": [-dev["G2"].values],
    "Average of G1 and G2": [-(dev["G1"] + dev["G2"]).values / 2],
    "Logistic, G2 and subject": lr_scores(["G2", "subject", "G2_x_subject"]),
    "Logistic, full Stage 2": lr_scores(stage2_cols),
}

rows = []
for name, score_list in candidates.items():
    for subj, mask in subject_masks.items():
        rows.append({
            "approach": name,
            "subject": subj,
            "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                      for s in score_list]),
            "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                               for s in score_list]),
        })

results_baselines = pd.DataFrame(rows).round(3)
results_baselines.to_csv("../outputs/tables/pooled_stronger_baselines.csv", index=False)
results_baselines

,approach,subject,avg_precision,precision_at_90_recall
0,"Rule, G2 only",All,0.877,0.741
1,"Rule, G2 only",Mathematics,0.933,0.845
2,"Rule, G2 only",Portuguese,0.799,0.436
3,Average of G1 and G2,All,0.879,0.672
4,Average of G1 and G2,Mathematics,0.918,0.762
5,Average of G1 and G2,Portuguese,0.834,0.583
6,"Logistic, G2 and subject",All,0.909,0.741
7,"Logistic, G2 and subject",Mathematics,0.943,0.845
8,"Logistic, G2 and subject",Portuguese,0.818,0.564
9,"Logistic, full Stage 2",All,0.920,0.769


In [10]:
# ===== Bootstrap gap in average precision, logistic against the plain average =====
rng = np.random.default_rng(RANDOM_STATE)

def bootstrap_gap(score_a, score_b, mask, n_boot=500):
    idx = np.where(mask)[0]
    gaps = []
    for _ in range(n_boot):
        sample = rng.choice(idx, size=len(idx), replace=True)
        if is_high[sample].sum() < 5:
            continue
        gaps.append(average_precision_score(is_high[sample], score_a[sample])
                    - average_precision_score(is_high[sample], score_b[sample]))
    return np.percentile(gaps, [2.5, 50, 97.5]).round(3)

full_lr = candidates["Logistic, full Stage 2"][0]
average = candidates["Average of G1 and G2"][0]

for subj in ["Mathematics", "Portuguese"]:
    low, mid, high = bootstrap_gap(full_lr, average, subject_masks[subj])
    print(f"{subj}: median gap {mid}, 95 percent interval {low} to {high}")

Mathematics: median gap 0.031, 95 percent interval 0.016 to 0.052
Portuguese: median gap 0.039, 95 percent interval 0.011 to 0.072
